In [1]:
# Justice Chowdary High School


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
            .config("spark.driver.memory", "2g") \
            .appName("justice_chowdary_highschool").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 07:30:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# JSON files - Schemas , 
# RDD s (low level) -> High level ( Dataframe API)

In [4]:
marks_path = "../data/jchs_marks_data.json"
weights_path = "../data/jchs_weightage.csv"

In [5]:
df_weights = spark.read\
                .format("csv")\
                .option("header",True)\
                .option("inferSchema",True)\
                .option("mode","PERMISSIVE")\
                .load(weights_path)

# instead of format(csv) and load ; we can directly use .csv(filepath)

In [59]:
df_weights.show(10)

+-----------+---------+
|  exam_name|weightage|
+-----------+---------+
|Unit Test 1|       10|
|Unit Test 2|       10|
|Unit Test 3|       10|
|Unit Test 4|       10|
|  Quarterly|       15|
|Half Yearly|       20|
|      Final|       25|
+-----------+---------+



In [9]:
df_weights.columns

['exam_name', 'weightage']

In [11]:
df_weights.count() , len(df_weights.columns)

(7, 2)

In [8]:
df_weights.rdd.getNumPartitions()

1

In [12]:
df_marks = spark.read\
                .format("json")\
                .option("multiline",True)\
                .load(marks_path)




In [13]:
df_marks.count()

1

In [23]:
df_marks.show()

+-------------+--------------------+--------------------+
|academic_year|              school|            students|
+-------------+--------------------+--------------------+
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|
+-------------+--------------------+--------------------+



In [15]:
df_marks.columns

['academic_year', 'school', 'students']

In [17]:
import pyspark.sql.functions as F

In [19]:
df2 = df_marks.withColumn("student_info", F.explode(F.col("students")))

In [22]:
df2.show(10)

+-------------+--------------------+--------------------+--------------------+
|academic_year|              school|            students|        student_info|
+-------------+--------------------+--------------------+--------------------+
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
|    2025-2026|Justice Chowdary ...|[{[{Unit Test 1, ...|{[{Unit Test 1, {...|
+-------------+--------------------+----------------

In [24]:
df2.count()

9

In [29]:
df3 = df2.withColumn("student_name",F.col("student_info.name"))\
            .withColumn("marks",F.col("student_info.marks"))\
            .drop("students","student_info")

In [30]:
df3.show(5)

+-------------+--------------------+------------+--------------------+
|academic_year|              school|student_name|               marks|
+-------------+--------------------+------------+--------------------+
|    2025-2026|Justice Chowdary ...|       Bhanu|[{Unit Test 1, {8...|
|    2025-2026|Justice Chowdary ...|     Bhargav|[{Unit Test 1, {7...|
|    2025-2026|Justice Chowdary ...|        Siva|[{Unit Test 1, {7...|
|    2025-2026|Justice Chowdary ...|        Teja|[{Unit Test 1, {8...|
|    2025-2026|Justice Chowdary ...|     Pradeep|[{Unit Test 1, {7...|
+-------------+--------------------+------------+--------------------+
only showing top 5 rows



In [34]:
df4 = df3.withColumn("marks_exp",F.explode("marks"))\
            .withColumn("exam_name",F.col("marks_exp.exam"))\
            .withColumn("subjects",F.col("marks_exp.subjects"))\
            .drop("marks","marks_exp")

In [35]:
df4.count()

63

In [37]:
df4.columns

['academic_year', 'school', 'student_name', 'exam_name', 'subjects']

In [42]:
df4.show(5,truncate=50)

+-------------+----------------------------+------------+-----------+------------------------+
|academic_year|                      school|student_name|  exam_name|                subjects|
+-------------+----------------------------+------------+-----------+------------------------+
|    2025-2026|Justice Chowdary High School|       Bhanu|Unit Test 1|{85, 72, 88, 82, 80, 78}|
|    2025-2026|Justice Chowdary High School|       Bhanu|Unit Test 2|{86, 75, 90, 84, 83, 81}|
|    2025-2026|Justice Chowdary High School|       Bhanu|Unit Test 3|{87, 74, 91, 85, 82, 79}|
|    2025-2026|Justice Chowdary High School|       Bhanu|Unit Test 4|{88, 76, 92, 86, 84, 82}|
|    2025-2026|Justice Chowdary High School|       Bhanu|  Quarterly|{89, 78, 93, 88, 86, 84}|
+-------------+----------------------------+------------+-----------+------------------------+
only showing top 5 rows



In [46]:
df5 = df4.select('academic_year', 'school', 'student_name', 
           'exam_name', 'subjects.English',
           'subjects.Hindi','subjects.Telugu',
           'subjects.Maths','subjects.Science',
          'subjects.Social')

In [48]:
df5.show(5)

+-------------+--------------------+------------+-----------+-------+-----+------+-----+-------+------+
|academic_year|              school|student_name|  exam_name|English|Hindi|Telugu|Maths|Science|Social|
+-------------+--------------------+------------+-----------+-------+-----+------+-----+-------+------+
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 1|     85|   72|    78|   88|     82|    80|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 2|     86|   75|    81|   90|     84|    83|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 3|     87|   74|    79|   91|     85|    82|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 4|     88|   76|    82|   92|     86|    84|
|    2025-2026|Justice Chowdary ...|       Bhanu|  Quarterly|     89|   78|    84|   93|     88|    86|
+-------------+--------------------+------------+-----------+-------+-----+------+-----+-------+------+
only showing top 5 rows



In [55]:
df6 = df5.withColumn("sub_total",F.coalesce(df5.English,F.lit(0))
                     + F.coalesce(df5.Hindi,F.lit(0)) + 
                     F.coalesce(df5.Telugu,F.lit(0)) +
                     F.coalesce(df5.Maths,F.lit(0)) +
                     F.coalesce(df5.Science,F.lit(0)) +
                     F.coalesce(df5.Social,F.lit(0))).\
                    select('academic_year', 'school', 'student_name', 
           'exam_name','sub_total')

In [57]:
df6.show(5)

+-------------+--------------------+------------+-----------+---------+
|academic_year|              school|student_name|  exam_name|sub_total|
+-------------+--------------------+------------+-----------+---------+
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 1|      485|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 2|      499|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 3|      498|
|    2025-2026|Justice Chowdary ...|       Bhanu|Unit Test 4|      508|
|    2025-2026|Justice Chowdary ...|       Bhanu|  Quarterly|      518|
+-------------+--------------------+------------+-----------+---------+
only showing top 5 rows



In [58]:
df_totals = df6.alias("df_totals")

In [61]:
df_weights.show(5)

+-----------+---------+
|  exam_name|weightage|
+-----------+---------+
|Unit Test 1|       10|
|Unit Test 2|       10|
|Unit Test 3|       10|
|Unit Test 4|       10|
|  Quarterly|       15|
+-----------+---------+
only showing top 5 rows



In [63]:
df_joined = df_totals.join(df_weights, "exam_name","left")

In [64]:
df_joined.show(5)

+-----------+-------------+--------------------+------------+---------+---------+
|  exam_name|academic_year|              school|student_name|sub_total|weightage|
+-----------+-------------+--------------------+------------+---------+---------+
|Unit Test 1|    2025-2026|Justice Chowdary ...|       Bhanu|      485|       10|
|Unit Test 2|    2025-2026|Justice Chowdary ...|       Bhanu|      499|       10|
|Unit Test 3|    2025-2026|Justice Chowdary ...|       Bhanu|      498|       10|
|Unit Test 4|    2025-2026|Justice Chowdary ...|       Bhanu|      508|       10|
|  Quarterly|    2025-2026|Justice Chowdary ...|       Bhanu|      518|       15|
+-----------+-------------+--------------------+------------+---------+---------+
only showing top 5 rows



In [65]:
df_weighted = df_joined.withColumn("weighted_score" , F.col("sub_total") * F.col("weightage") / 100 )

In [66]:
df_weighted.show(5)

+-----------+-------------+--------------------+------------+---------+---------+--------------+
|  exam_name|academic_year|              school|student_name|sub_total|weightage|weighted_score|
+-----------+-------------+--------------------+------------+---------+---------+--------------+
|Unit Test 1|    2025-2026|Justice Chowdary ...|       Bhanu|      485|       10|          48.5|
|Unit Test 2|    2025-2026|Justice Chowdary ...|       Bhanu|      499|       10|          49.9|
|Unit Test 3|    2025-2026|Justice Chowdary ...|       Bhanu|      498|       10|          49.8|
|Unit Test 4|    2025-2026|Justice Chowdary ...|       Bhanu|      508|       10|          50.8|
|  Quarterly|    2025-2026|Justice Chowdary ...|       Bhanu|      518|       15|          77.7|
+-----------+-------------+--------------------+------------+---------+---------+--------------+
only showing top 5 rows



In [69]:
df_prefinal = df_weighted.groupBy("student_name").agg(F.round(F.sum("weighted_score"),2).alias("final_score"))

In [70]:
df_prefinal.show(5)

+------------+-----------+
|student_name|final_score|
+------------+-----------+
|      Ramesh|      482.6|
|     Bhargav|      472.6|
|      Suresh|      442.2|
|      Ganesh|      412.6|
|        Teja|      536.7|
+------------+-----------+
only showing top 5 rows



In [71]:
from pyspark.sql.window import Window

window1 = Window.orderBy(F.col("final_score").desc())

df_final = df_prefinal.withColumn("rank",F.row_number().over(window1))

In [73]:
df_final.show()

+------------+-----------+----+
|student_name|final_score|rank|
+------------+-----------+----+
|        Teja|      536.7|   1|
|         Sai|      519.6|   2|
|       Bhanu|      516.2|   3|
|      Ramesh|      482.6|   4|
|     Bhargav|      472.6|   5|
|     Pradeep|      458.2|   6|
|      Suresh|      442.2|   7|
|        Siva|     429.85|   8|
|      Ganesh|      412.6|   9|
+------------+-----------+----+



26/08/22 08:32:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:32:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:32:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:32:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:32:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:32:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 0

In [74]:
df_final.repartition(1).write.csv("../data/final_ranks.csv")

26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 08:34:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/22 0

In [75]:
import pandas as pd

In [77]:
df_final.toPandas().to_csv("../data/final_ranks2.csv",index = False)

26/08/22 09:02:17 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-028be59b-e54a-45df-88e6-6b4c2d1cda60. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-028be59b-e54a-45df-88e6-6b4c2d1cda60
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize